# CFDFlow-Agent: CFD 后处理智能分析 Agent

基于 `hello-agents` 框架构建的 **CFD 仿真后处理分析 Agent**，支持：

| 亮点 | 说明 |
|------|------|
| **专业工具路由** | Agent 根据列名和用户问题自动选择残差收敛、升阻力、压力分布、速度剖面、网格无关性分析工具 |
| **确定性计算** | 残差下降阶数、Cl/Cd 均值与波动、Cp 分布、边界层速度剖面、Richardson 外推 & GCI 由 Python 工具完成 |
| **结构化报告** | 报告固定包含算例概况、收敛判断、关键气动指标、图表、异常点、工程解释、局限性 |
| **收敛性保护** | 当残差未收敛或升阻力强烈振荡时，Agent 必须提示"当前结果不宜作为稳态结论" |
| **可追踪性** | 保存 `trace.json`、`report.md`、`figures/`，便于展示 Agent 每一步推理 |

## 环境配置

1. 确保已激活环境 
2. 安装依赖：`pip install -r requirements.txt`
3. 复制 `.env.example` 为 `.env` 并填入 API Key
4. 将本 notebook 的 Kernel 设置为对应环境的 Python 解释器

In [ ]:
import sys
import os
from pathlib import Path

# 确保项目根目录在 Python 路径中
PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# 同时确保 hello-agents 框架可用
#FRAMEWORK_ROOT = PROJECT_ROOT.parent.parent
#if str(FRAMEWORK_ROOT) not in sys.path:
#    sys.path.insert(0, str(FRAMEWORK_ROOT))

from cfd_analysis_agent import run_analysis
from cfd_analysis_agent.presentation import render_trace_table, render_full_report, render_diagnostics
from cfd_analysis_agent.agent_runner import build_plaintext_event_handler

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

print(f"项目根目录: {PROJECT_ROOT}")
print(f"数据目录:   {DATA_DIR}")
print(f"输出目录:   {OUTPUT_DIR}")
print(f"Python:     {sys.executable}")

In [2]:
from IPython.display import display, Markdown, HTML

def render_run_summary(result):
    """渲染一次分析运行的摘要信息。"""
    lines = [
        f"| 项目 | 值 |",
        f"|------|----|",
        f"| 数据文件 | `{result.data_context.data_path}` |",
        f"| 数据规模 | {result.data_context.shape[0]} 行 × {result.data_context.shape[1]} 列 |",
        f"| CFD 任务类型 | `{result.cfd_task_type}` |",
        f"| 使用工具 | {', '.join(result.tools_used)} |",
        f"| 推理步数 | {len(result.step_traces)} |",
        f"| 工作流完成 | {'✅' if result.workflow_complete else '❌'} |",
        f"| 质量模式 | {result.quality_mode} |",
        f"| 审稿状态 | {result.review_status} |",
        f"| 总耗时 | {result.total_duration_ms / 1000:.1f}s |",
        f"| 运行目录 | `{result.run_dir}` |",
    ]
    display(Markdown("### 运行摘要\n\n" + "\n".join(lines)))

---
## Demo 1: 残差收敛分析

分析 CFD 求解器的残差历史数据，判断计算是否收敛。

**预期行为**：Agent 识别到 `continuity`, `x-velocity` 等残差列名 → 自动路由到 `ResidualAnalysisTool` → 计算各残差下降阶数 → 生成半对数残差图 → 给出收敛判定。

In [ ]:
result_residual = run_analysis(
    DATA_DIR / "residual_history.csv",
    query="请分析此 CFD 仿真的残差收敛情况，判断计算是否达到稳态收敛标准。",
    output_dir=OUTPUT_DIR,
    quality_mode="draft",
    max_steps=6,
    verbose=True,
)

render_run_summary(result_residual)

In [ ]:
display(render_trace_table(result_residual))

---
## Demo 2: 升阻力系数分析

分析气动力系数（Cl/Cd/Cm）的时间历程，检测振荡和稳定性。

**预期行为**：Agent 识别到 `Cl`, `Cd`, `Cm` 列名 → 自动路由到 `ForceAnalysisTool` → 计算均值、标准差、振荡检测 → 生成力系数时间历程图 → 给出稳定性判定。

In [ ]:
result_force = run_analysis(
    DATA_DIR / "force_coefficients.csv",
    query="请分析此 CFD 算例的升力系数(Cl)、阻力系数(Cd)和俯仰力矩系数(Cm)的收敛性和统计特征。",
    output_dir=OUTPUT_DIR,
    quality_mode="draft",
    max_steps=6,
    verbose=True,
)

render_run_summary(result_force)

In [ ]:
display(render_full_report(result_force))

---
## Demo 3: 压力分布分析

分析翼型表面压力系数 Cp 的分布，识别吸力峰值和驻点。

**预期行为**：Agent 识别到 `x/c`, `Cp_upper`, `Cp_lower` 列名 → 自动路由到 `PressureAnalysisTool` → 识别吸力峰值位置和大小 → 生成 Cp vs x/c 图（y 轴反转） → 给出气动特征分析。

In [ ]:
result_pressure = run_analysis(
    DATA_DIR / "pressure_distribution.csv",
    query="请分析此翼型表面的压力分布(Cp)，识别吸力峰值位置、驻点位置，并评估气动载荷分布特征。",
    output_dir=OUTPUT_DIR,
    quality_mode="draft",
    max_steps=6,
    verbose=True,
)

render_run_summary(result_pressure)

---
## Demo 4: 速度剖面分析

分析边界层速度剖面，与壁面律/对数律理论曲线对比。

**预期行为**：Agent 识别到 `y_plus`, `u_plus` 列名 → 自动路由到 `VelocityAnalysisTool` → 与 law of wall / log law 对比 → 估算边界层厚度参数 → 生成速度剖面图。

In [ ]:
result_velocity = run_analysis(
    DATA_DIR / "velocity_profile.csv",
    query="请分析此边界层速度剖面数据，与理论壁面律/对数律对比，评估边界层特征和湍流模型表现。",
    output_dir=OUTPUT_DIR,
    quality_mode="draft",
    max_steps=6,
    verbose=True,
)

render_run_summary(result_velocity)

---
## Demo 5: 网格无关性分析

基于多级网格的计算结果，进行 Richardson 外推和 GCI 分析。

**预期行为**：Agent 识别到 `cell_count`, `mesh_level` 等列名 → 自动路由到 `GridStudyTool` → 执行 Richardson 外推、GCI 计算 → 生成网格收敛图 → 报告数值不确定度。

In [ ]:
result_grid = run_analysis(
    DATA_DIR / "mesh_study.csv",
    query="请对此多级网格计算结果进行网格无关性分析，包括 Richardson 外推、GCI 数值不确定度评估。",
    output_dir=OUTPUT_DIR,
    quality_mode="draft",
    max_steps=6,
    verbose=True,
)

render_run_summary(result_grid)

In [ ]:
display(render_full_report(result_grid))

In [ ]:
# 查看所有 Demo 的错误诊断
for name, result in [
    ("残差收敛", result_residual),
    ("升阻力", result_force),
    ("压力分布", result_pressure),
    ("速度剖面", result_velocity),
    ("网格无关性", result_grid),
]:
    print(f"\n{'='*60}")
    print(f"  {name} | workflow_complete={result.workflow_complete} | tools={result.tools_used}")
    print(f"{'='*60}")
    display(render_diagnostics(result))

---
## 项目亮点

### 1. LangGraph 状态图编排：
将 CFD 后处理中可确定计算的残差收敛、升阻力系数、压力分布、速度剖面、网格无关性分析封装为专用 tools，使 LLM 负责路由和解释，数值由工具层计算。

### 2. 上下文工程控制：
通过滑动窗口+摘要压缩的长短期记忆处理方式，用结构化 facts 保存数据形状、任务类型、工具结果、warning 和图表路径，再用最近 N 轮原始消息补充证据，降低长上下文噪声和数值幻觉风险。 

### 3.确定性 CFD 工具链：
封装残差收敛、升阻力、压力分布、速度剖面、网格无关性等专业分析工具，关键数值结论由封装Tool计算生成，LLM 只负责解释和报告组织。

### 4. Gradio 可视化工作台：
增加工作流程交互式 Demo，支持上传 CSV/Excel/PDF、配置上下文窗口、查看节点执行表、最终报告、图表、结构化事实和 trace，提升项目可演示性。

### 5. 完整可追踪性
每次运行生成独立的 `run_YYYYMMDD_HHMMSS/` 目录，包含：
- `data/cleaned_data.csv` — 清洗后的数据
- `figures/` — 所有图表
- `logs/agent_trace.json` — 完整推理轨迹
- `final_report.md` — 最终报告